## Imports

In [ ]:
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler, FunctionTransformer
from sklearn.impute import SimpleImputer

## Load Data

In [52]:
df = pd.read_csv("../raw_data/hotel_bookings.csv")

## Engineer Numerical Features

In [ ]:
def engineer_numerical_features(X: pd.DataFrame) -> pd.DataFrame:
    """
    Create engineered features from the raw hotel booking dataset:
    - removes the target if it is present
    - removes known leakage columns
    - converts ID-like columns into binary indicators
    """
    X = X.copy()

    # Drop target if it is accidentally included
    if "is_canceled" in X.columns:
        X = X.drop(columns="is_canceled")

    # Drop leakage columns: these contain future information
    leakage_cols = ["reservation_status", "reservation_status_date"]
    cols_to_drop = [col for col in leakage_cols if col in X.columns]
    if cols_to_drop:
        X = X.drop(columns=cols_to_drop)

    # company ID -> binary flag
    if "company" in X.columns:
        X["company_booking"] = (X["company"] != 0).astype(int)
        X = X.drop(columns="company")

    # agent ID -> binary flag
    if "agent" in X.columns:
        X["has_agent"] = (X["agent"] != 0).astype(int)
        X = X.drop(columns="agent")

    # parking spaces -> simpler yes/no feature
    if "required_car_parking_spaces" in X.columns:
        X["has_parking"] = (X["required_car_parking_spaces"] > 0).astype(int)
        X = X.drop(columns="required_car_parking_spaces")

    return X

## Preprocess Pipeline

In [ ]:
def build_numerical_preprocessor(numeric_features, binary_features):
    """
    Build a ColumnTransformer for numerical preprocessing:
    numeric_features : Continuous or count-based numerical columns to impute and scale
    binary_features : Binary 0/1 columns to impute only
    """
    # Continuous / count numerical features:
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", RobustScaler())
    ])

    # Binary features:
    binary_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent"))
    ])

    # Preprocessor:
    preprocessor = ColumnTransformer([
        ("num", numeric_pipeline, numeric_features),
        ("bin", binary_pipeline, binary_features)
    ])

    # Pipeline:
    pipeline = Pipeline([
        ("feature_engineering", FunctionTransformer(engineer_features, validate=False)),
        ("preprocessing", preprocessor)
    ])

    return pipeline
